# 🌙 Sleep Stage Classification: Machine Learning Guide 

## 🛠 0. เตรียมเครื่องมือ (Setup)


In [ ]:
# นำเข้าเครื่องมือมาตรฐานสำหรับงาน Data Science
import os
import glob
import pandas as pd # สำหรับจัดการข้อมูลตาราง
import numpy as np  # สำหรับการคำนวณทางคณิตศาสตร์
import lightgbm as lgb # โมเดลที่เชียวชาญด้านตาราง
from scipy.stats import skew, kurtosis # ใช้หาลักษณะความเบี้ยวของสัญญาณ
from sklearn.model_selection import StratifiedGroupKFold # สำหรับแบ่งชุดข้อมูลทดสอบ
from sklearn.metrics import accuracy_score # วัดผลความแม่นยำ
from tqdm.notebook import tqdm # แถบแสดงความคืบหน้าการทำงาน
import warnings

# ปิดข้อความแจ้งเตือนที่ไม่จำเป็น
warnings.filterwarnings('ignore')


## ⚙️ 1. สกัดฟีเจอร์ (Feature Engineering V2)
เราจะเปลี่ยนข้อมูลดิบ 480 แถว ให้กลายเป็นตัวเลขสถิติชุดเดียวที่บอกลักษณะของช่วงเวลานั้นๆ ครับ


In [ ]:
def extract_features(df_segment):
    """
    ฟังก์ชันสำหรับเปลี่ยนสัญญาณ 480 แถว (30 วินาที) ให้กลายเป็นสรุปสถิติ
    """
    features = {}
    
    # คำนวณความแรงสั่นสะเทือนรวม (ACC Magnitude)
    df_segment['ACC_Mag'] = np.sqrt(df_segment['ACC_X']**2 + df_segment['ACC_Y']**2 + df_segment['ACC_Z']**2)
    
    # รายชื่อคอลัมน์ข้อมูลที่มี
    cols_to_extract = ['BVP', 'ACC_X', 'ACC_Y', 'ACC_Z', 'ACC_Mag', 'TEMP', 'EDA', 'HR', 'IBI']
    
    for col in cols_to_extract:
        # เติมค่าว่างด้วยค่าก่อนหน้าหรือค่าถัดไป
        series = df_segment[col].fillna(method='ffill').fillna(method='bfill').fillna(0)
        
        # [1] ค่าสถิติพื้นฐาน
        features[f'{col}_mean'] = series.mean()
        features[f'{col}_std'] = series.std()
        features[f'{col}_min'] = series.min()
        features[f'{col}_max'] = series.max()
        features[f'{col}_median'] = series.median()
        features[f'{col}_skew'] = skew(series)     # ความเบี้ยว
        features[f'{col}_kurt'] = kurtosis(series) # ความโด่ง
        features[f'{col}_rms'] = np.sqrt(np.mean(series**2)) # พลังงานสัญญาณ
        
        # [2] การกระจายตัว (Quartiles)
        q25 = np.percentile(series, 25)
        q75 = np.percentile(series, 75)
        features[f'{col}_q25'] = q25
        features[f'{col}_q75'] = q75
        features[f'{col}_range'] = features[f'{col}_max'] - features[f'{col}_min']
        features[f'{col}_iqr'] = q75 - q25
        
        # [3] วิเคราะห์แยกครึ่งรอบ (Sub-Epoch) เพื่อดูแนวโน้ม (Trend)
        half = len(series) // 2
        series_h1 = series.iloc[:half]
        series_h2 = series.iloc[half:]
        
        features[f'{col}_mean_h1'] = series_h1.mean()
        features[f'{col}_std_h1'] = series_h1.std()
        features[f'{col}_mean_h2'] = series_h2.mean()
        features[f'{col}_std_h2'] = series_h2.std()
        features[f'{col}_trend'] = features[f'{col}_mean_h2'] - features[f'{col}_mean_h1']
        
    return features


## 📂 2. การโหลดข้อมูลและทำ Cache
เราจะเซฟผลลัพธ์ลงไฟล์ `train_features_v2.csv` เพื่อที่รอบหน้าจะได้รันได้เร็วขึ้นครับ


In [ ]:
TRAIN_DIR = './data/train/train'
TEST_DIR = './data/test_segment/test_segment'

def process_train_data():
    all_features = []
    all_labels = []
    groups = [] 
    
    train_files = glob.glob(os.path.join(TRAIN_DIR, '*.csv'))
    
    for file_id, filepath in enumerate(tqdm(train_files, desc="ประมวลผลไฟล์ Train")):
        df = pd.read_csv(filepath)
        num_epochs = len(df) // 480
        
        for i in range(num_epochs):
            start_idx = i * 480
            end_idx = start_idx + 480
            segment = df.iloc[start_idx:end_idx].copy()
            
            label = segment['Sleep_Stage'].iloc[0] # ค้นหาว่าช่วงนี้คือคลาสไหน
            feats = extract_features(segment)
            
            all_features.append(feats)
            all_labels.append(label)
            groups.append(file_id) 
            
    df_features = pd.DataFrame(all_features)
    df_features['label'] = all_labels
    df_features['group'] = groups
    return df_features

# เช็คว่ามี Cache ไหม ถ้าไม่มีก็สร้างใหม่
if os.path.exists('train_features_v2.csv'):
    print("💡 พบไฟล์ Cache เดิม! กำลังโหลด...")
    train_df = pd.read_csv('train_features_v2.csv')
else:
    train_df = process_train_data()
    train_df.to_csv('train_features_v2.csv', index=False)

print(f"จำนวน Epoch ทั้งหมด: {len(train_df)}")


## ⏳ 3. บริบทของเวลา (Temporal Context)
เชื่อมโยงความสัมพันธ์ของวินาทีก่อนหน้าและถัดไป


In [ ]:
def add_temporal_features(df):
    df = df.copy()
    # ยกเว้นคอลัมน์ที่ไม่ใช่ข้อมูลสัญญาณ
    feature_cols = [c for c in df.columns if c not in ['label', 'group', 'id']]
    
    # ดึงค่าของรอบที่แล้ว และรอบถัดไป
    lag1 = df.groupby('group')[feature_cols].shift(1).add_suffix('_lag1')
    lead1 = df.groupby('group')[feature_cols].shift(-1).add_suffix('_lead1')
    lag2 = df.groupby('group')[feature_cols].shift(2).add_suffix('_lag2')
    lead2 = df.groupby('group')[feature_cols].shift(-2).add_suffix('_lead2')
    
    # รวมข้อมูลเข้าด้วยกันและเติมค่าว่างที่เกิดจากการ Shift
    df = pd.concat([df, lag1, lead1, lag2, lead2], axis=1)
    df.fillna(method='bfill', inplace=True)
    df.fillna(method='ffill', inplace=True)
    
    # คำนวณความต่าง (Derivative) เพื่อดูความเร็วการเปลี่ยนแปลง
    for col in feature_cols:
        df[f'{col}_diff_lag1'] = df[col] - df[f'{col}_lag1']
        df[f'{col}_diff_lead1'] = df[f'{col}_lead1'] - df[col]
        
    return df

print("✨ กำลังสร้างมิติเวลาและค่าความแตกต่าง...")
train_df = add_temporal_features(train_df)


## 🧠 4. ฝึกสอนโมเดล (Training)
ใช้ LightGBM พร้อมปรับน้ำหนักคลาสให้สมดุลเพื่อคะแนนสูงสุดใน V2 ครับ


In [ ]:
features = [c for c in train_df.columns if c not in ['label', 'group']]
X = train_df[features]

# แปลง Label ตัวอักษรเป็นตัวเลข
y_map = {'W':0, 'N1':1, 'N2':2, 'N3':3, 'R':4}
inv_y_map = {v:k for k,v in y_map.items()}
y = train_df['label'].map(y_map)
groups = train_df['group']

# แบ่งกลุ่มสอบ 5 รอบ
sgkf = StratifiedGroupKFold(n_splits=5)
models = []

lgb_params = {
    'objective': 'multiclass',
    'num_class': 5,
    'learning_rate': 0.03,
    'num_leaves': 127,
    'class_weight': 'balanced', # ช่วยคลาสหายาก (N1) ให้มีตัวตน
    'n_estimators': 2500,
    'device': 'cpu',
    'verbose': -1
}

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups)):
    print(f"--- 🏋️‍♂️ Fold {fold+1} ---")
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(150, verbose=False)])
    models.append(model)


## 🧪 5. ทำนายชุดคำตอบ (Test Prediction)


In [ ]:
def process_test_data():
    all_features = []
    test_folders = glob.glob(os.path.join(TEST_DIR, '*'))
    
    for folder in tqdm(test_folders, desc="ทำนายผล Test"):
        segment_files = sorted(glob.glob(os.path.join(folder, '*.csv')))
        
        folder_features = []
        folder_ids = []
        for filepath in segment_files:
            segment_id = os.path.basename(filepath).replace('.csv', '')
            df = pd.read_csv(filepath)
            feats = extract_features(df)
            folder_features.append(feats)
            folder_ids.append(segment_id)
            
        df_folder_feats = pd.DataFrame(folder_features)
        df_folder_feats['id'] = folder_ids
        df_folder_feats['group'] = os.path.basename(folder)
        df_folder_feats = add_temporal_features(df_folder_feats)
        all_features.append(df_folder_feats)
        
    return pd.concat(all_features, ignore_index=True)

test_df = process_test_data()


## 🏁 6. สร้างไฟล์ส่งคำตอบ


In [ ]:
test_X = test_df[features]
test_preds = np.zeros((len(test_X), 5)) 

# โหวตจากโมเดลทั้ง 5 ตัว
for model in models:
    test_preds += model.predict_proba(test_X) / 5

# ทำนายคลาสที่ความน่าจะเป็นสูงสุด
final_preds_idx = np.argmax(test_preds, axis=1)
final_preds_labels = [inv_y_map[idx] for idx in final_preds_idx]

submission = pd.DataFrame({'id': test_df['id'], 'labels': final_preds_labels})
sample_sub = pd.read_csv('./data/sample_submission.csv')
submission = submission.set_index('id').reindex(sample_sub['id']).reset_index()

submission.to_csv('submission_v2_final.csv', index=False)
print("🔥 เรียบร้อย! นำไฟล์ [ submission_v2_final.csv ] ไปใช้ได้เลยครับ!")
